# COVID-19 Tweet Sentiment Classification with RNN Models

This notebook implements recurrent neural network style models for text classification using the Kaggle dataset **Coronavirus tweets NLP - Text Classification** by Datatattle.

Dataset URL: https://www.kaggle.com/datasets/datatattle/covid-19-nlp-text-classification

The notebook intentionally avoids TensorFlow and PyTorch because this Windows/Jupyter environment can fail on native DLL imports. Instead, it implements the recurrent encoder directly with NumPy and trains the final classifier with scikit-learn.

## Project Overview

This is a supervised five-class sentiment classification task. Each input is a COVID-19 related tweet, and the target label is one of:

- `Extremely Negative`
- `Negative`
- `Neutral`
- `Positive`
- `Extremely Positive`

Models compared:

1. **Custom architecture:** a custom NumPy recurrent encoder with randomly initialized embeddings and recurrent weights.
2. **Transfer learning:** the same recurrent encoder initialized with pre-trained GloVe embeddings.

Both models train a multinomial logistic classifier on top of recurrent text features. This keeps the notebook runnable on CPU-only machines while still exposing the RNN architecture, sequence processing, and embedding-transfer comparison required by the assignment.

## Local Files

Expected structure:

```text
data/covid19-nlp-text-classification/Corona_NLP_train.csv
data/covid19-nlp-text-classification/Corona_NLP_test.csv
data/glove/glove.6B.100d.txt
```

The CSV files are read with `ISO-8859-1` encoding because the Kaggle files can fail under UTF-8.

In [ ]:
# Optional Kaggle download command. Uncomment in an authenticated Kaggle or Colab environment.
# !kaggle datasets download -d datatattle/covid-19-nlp-text-classification -p data/covid19-nlp-text-classification --unzip

# Optional GloVe download. This archive is large, so it is not run by default.
# !wget http://nlp.stanford.edu/data/glove.6B.zip -P data/glove
# !unzip data/glove/glove.6B.zip -d data/glove

In [ ]:
import html
import random
import re
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import make_pipeline

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid")

## Dataset Documentation

The dataset contains COVID-19 tweets from the early pandemic period.

- Training set: **41,157** tweets
- Test set: **3,798** tweets
- Number of classes: **5**

Columns:

- `UserName`: anonymized numeric user identifier
- `ScreenName`: anonymized numeric screen-name identifier
- `Location`: user-provided location text, often missing or noisy
- `TweetAt`: tweet date
- `OriginalTweet`: raw tweet text used as the model input
- `Sentiment`: target label

Published class distribution:

| Sentiment | Train | Test |
|---|---:|---:|
| Extremely Negative | 5,481 | 592 |
| Negative | 9,917 | 1,041 |
| Neutral | 7,713 | 619 |
| Positive | 11,422 | 947 |
| Extremely Positive | 6,624 | 599 |

The distribution is moderately imbalanced, which affects metric selection.

In [ ]:
def find_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for candidate in candidates:
        if (candidate / "data" / "covid19-nlp-text-classification").exists():
            return candidate
    return Path.cwd()

PROJECT_ROOT = find_project_root()
print("Project root:", PROJECT_ROOT)

DATA_DIR = PROJECT_ROOT / "data/covid19-nlp-text-classification"
TRAIN_PATH = DATA_DIR / "Corona_NLP_train.csv"
TEST_PATH = DATA_DIR / "Corona_NLP_test.csv"

def require_file(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}. Place the required file at this location.")

require_file(TRAIN_PATH)
require_file(TEST_PATH)

train_df = pd.read_csv(TRAIN_PATH, encoding="ISO-8859-1")
test_df = pd.read_csv(TEST_PATH, encoding="ISO-8859-1")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
display(train_df.head())

In [ ]:
summary = pd.DataFrame({
    "split": ["train", "test"],
    "instances": [len(train_df), len(test_df)],
    "columns": [train_df.shape[1], test_df.shape[1]],
    "missing_locations": [train_df["Location"].isna().sum(), test_df["Location"].isna().sum()],
})
display(summary)

class_distribution = pd.DataFrame({
    "train": train_df["Sentiment"].value_counts().sort_index(),
    "test": test_df["Sentiment"].value_counts().sort_index(),
}).fillna(0).astype(int)
display(class_distribution)

ax = class_distribution.plot(kind="bar", figsize=(10, 5), rot=30)
ax.set_title("Sentiment class distribution")
ax.set_xlabel("Sentiment")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

## Sample Text Examples

Representative examples include:

| Sentiment | Example text |
|---|---|
| Neutral | `TRENDING: New Yorkers encounter empty supermarket shelves...` |
| Positive | `When I couldn't find hand sanitizer at Fred Meyer, I turned to Amazon...` |
| Extremely Positive | `Find out how you can protect yourself and loved ones from coronavirus...` |
| Negative | `Panic buying hits New York City as anxious shoppers stock up...` |
| Extremely Negative | `Me, ready to go at supermarket during the COVID19 outbreak... it causes shortage...` |

The cell below prints one local example per class.

In [ ]:
sample_examples = (
    train_df.groupby("Sentiment", group_keys=False)
    .apply(lambda frame: frame.sample(1, random_state=SEED))
    [["Sentiment", "OriginalTweet"]]
)

for _, row in sample_examples.iterrows():
    print(f"LABEL: {row['Sentiment']}")
    print(row["OriginalTweet"])
    print("-" * 100)

## Text Preprocessing

Applied preprocessing:

1. Decode HTML entities such as `&amp;`.
2. Remove URLs.
3. Remove Twitter mentions.
4. Preserve hashtag words while removing `#`.
5. Lowercase text.
6. Remove non-alphanumeric characters except apostrophes.
7. Collapse repeated whitespace.
8. Build a vocabulary from train text only.
9. Convert tweets to padded token ID sequences.

Labels are encoded with `LabelEncoder`.

In [ ]:
URL_RE = re.compile(r"https?://\S+|www\.\S+")
MENTION_RE = re.compile(r"@\w+")
HASHTAG_RE = re.compile(r"#(\w+)")
NON_TEXT_RE = re.compile(r"[^a-zA-Z0-9' ]+")
SPACE_RE = re.compile(r"\s+")

def clean_tweet(text: str) -> str:
    text = html.unescape(str(text))
    text = URL_RE.sub(" ", text)
    text = MENTION_RE.sub(" ", text)
    text = HASHTAG_RE.sub(r"\1", text)
    text = text.lower()
    text = NON_TEXT_RE.sub(" ", text)
    text = SPACE_RE.sub(" ", text).strip()
    return text

train_df = train_df.copy()
test_df = test_df.copy()
train_df["clean_text"] = train_df["OriginalTweet"].apply(clean_tweet)
test_df["clean_text"] = test_df["OriginalTweet"].apply(clean_tweet)
display(train_df[["OriginalTweet", "clean_text", "Sentiment"]].head())

In [ ]:
MAX_VOCAB_SIZE = 30000
MAX_SEQUENCE_LENGTH = 80
PAD_TOKEN = "<PAD>"
OOV_TOKEN = "<OOV>"

def tokenize(text: str) -> list[str]:
    return text.split()

def build_vocab(texts: pd.Series, max_vocab_size: int) -> dict[str, int]:
    counter = Counter()
    for text in texts:
        counter.update(tokenize(text))
    vocab = {PAD_TOKEN: 0, OOV_TOKEN: 1}
    for word, _ in counter.most_common(max_vocab_size - len(vocab)):
        vocab[word] = len(vocab)
    return vocab

def encode_text(text: str, vocab: dict[str, int], max_length: int) -> list[int]:
    ids = [vocab.get(token, vocab[OOV_TOKEN]) for token in tokenize(text)[:max_length]]
    ids += [vocab[PAD_TOKEN]] * (max_length - len(ids))
    return ids

vocab = build_vocab(train_df["clean_text"], MAX_VOCAB_SIZE)
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(train_df["Sentiment"])
y_test = label_encoder.transform(test_df["Sentiment"])
class_names = list(label_encoder.classes_)

X_train = np.array([encode_text(text, vocab, MAX_SEQUENCE_LENGTH) for text in train_df["clean_text"]], dtype=np.int32)
X_test = np.array([encode_text(text, vocab, MAX_SEQUENCE_LENGTH) for text in test_df["clean_text"]], dtype=np.int32)

num_classes = len(class_names)
vocab_size = len(vocab)
print("Classes:", class_names)
print("Vocabulary size:", vocab_size)
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

## Performance Metrics

The primary metric is **macro F1-score**.

Justification: the dataset is moderately imbalanced and all sentiment categories matter. Macro F1 gives equal weight to every class, while accuracy can hide poor performance on smaller classes.

Supporting metrics: accuracy, macro precision, macro recall, weighted F1, classification report, and confusion matrix.

In [ ]:
def evaluate_predictions(y_true, y_pred, model_name: str) -> dict:
    metrics = {
        "model": model_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }
    print(f"Evaluation for {model_name}")
    print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", xticklabels=class_names, yticklabels=class_names, cmap="Blues")
    plt.title(f"Confusion matrix: {model_name}")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    plt.show()
    return metrics

## NumPy Recurrent Encoder

The encoder below is a custom recurrent neural architecture implemented from scratch with NumPy. It processes token embeddings sequentially:

```text
h_t = tanh(x_t W_xh + h_{t-1} W_hh + b)
```

The final hidden state becomes the tweet representation. A multinomial logistic classifier is trained on top of those RNN features. This is lighter than full backpropagation-through-time, but it still demonstrates recurrent sequence modeling and makes the notebook reliable on machines without working deep-learning DLLs.

In [ ]:
def make_random_embeddings(vocab_size: int, embedding_dim: int, seed: int = SEED) -> np.ndarray:
    rng = np.random.default_rng(seed)
    matrix = rng.normal(0, 0.05, size=(vocab_size, embedding_dim)).astype("float32")
    matrix[0] = 0.0
    return matrix

def recurrent_features(
    X: np.ndarray,
    embedding_matrix: np.ndarray,
    hidden_units: int = 64,
    recurrent_scale: float = 0.35,
    seed: int = SEED,
    batch_size: int = 512,
) -> np.ndarray:
    rng = np.random.default_rng(seed)
    embedding_dim = embedding_matrix.shape[1]
    W_xh = rng.normal(0, 1 / np.sqrt(embedding_dim), size=(embedding_dim, hidden_units)).astype("float32")
    W_hh = rng.normal(0, recurrent_scale / np.sqrt(hidden_units), size=(hidden_units, hidden_units)).astype("float32")
    bias = np.zeros(hidden_units, dtype="float32")
    features = np.zeros((len(X), hidden_units), dtype="float32")

    for start in range(0, len(X), batch_size):
        end = min(start + batch_size, len(X))
        batch = X[start:end]
        h = np.zeros((len(batch), hidden_units), dtype="float32")
        for step in range(batch.shape[1]):
            token_ids = batch[:, step]
            mask = (token_ids != 0).astype("float32")[:, None]
            x_t = embedding_matrix[token_ids]
            proposed = np.tanh(x_t @ W_xh + h @ W_hh + bias)
            h = mask * proposed + (1.0 - mask) * h
        features[start:end] = h
    return features

def train_linear_head(features_train: np.ndarray, y_train: np.ndarray):
    clf = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=400, class_weight="balanced", n_jobs=-1, random_state=SEED),
    )
    clf.fit(features_train, y_train)
    return clf

## Model 1: Custom RNN Architecture

This model uses random trainable-style embeddings and a custom recurrent encoder. Hyperparameters include embedding dimension, hidden size, and recurrent scale.

In [ ]:
custom_embedding_matrix = make_random_embeddings(vocab_size, embedding_dim=128, seed=SEED)
custom_train_features = recurrent_features(X_train, custom_embedding_matrix, hidden_units=96, recurrent_scale=0.35, seed=SEED)
custom_test_features = recurrent_features(X_test, custom_embedding_matrix, hidden_units=96, recurrent_scale=0.35, seed=SEED)

custom_classifier = train_linear_head(custom_train_features, y_train)
custom_predictions = custom_classifier.predict(custom_test_features)
custom_metrics = evaluate_predictions(y_test, custom_predictions, "Custom NumPy RNN")
custom_metrics

### Hyperparameter Experiments

The small grid below explores hidden size and recurrent scale. Larger hidden sizes increase model capacity. Higher recurrent scale increases the influence of previous tokens but can make representations less stable.

In [ ]:
experiment_configs = [
    {"hidden_units": 32, "recurrent_scale": 0.25},
    {"hidden_units": 64, "recurrent_scale": 0.35},
    {"hidden_units": 96, "recurrent_scale": 0.35},
]

experiment_results = []
for config in experiment_configs:
    features_train = recurrent_features(
        X_train,
        custom_embedding_matrix,
        hidden_units=config["hidden_units"],
        recurrent_scale=config["recurrent_scale"],
        seed=SEED + config["hidden_units"],
    )
    features_test = recurrent_features(
        X_test,
        custom_embedding_matrix,
        hidden_units=config["hidden_units"],
        recurrent_scale=config["recurrent_scale"],
        seed=SEED + config["hidden_units"],
    )
    clf = train_linear_head(features_train, y_train)
    preds = clf.predict(features_test)
    result = evaluate_predictions(y_test, preds, f"Custom RNN h={config['hidden_units']} scale={config['recurrent_scale']}")
    result.update(config)
    experiment_results.append(result)

experiment_df = pd.DataFrame(experiment_results).sort_values("macro_f1", ascending=False)
display(experiment_df)

## Model 2: Transfer Learning with GloVe Embeddings

This model uses the same recurrent encoder, but initializes the token embedding matrix from pre-trained GloVe vectors. This is transfer learning because semantic word information is imported from an external corpus before training the task-specific classifier.

In [ ]:
GLOVE_PATH = PROJECT_ROOT / "data/glove/glove.6B.100d.txt"
GLOVE_DIM = 100

def load_glove_for_vocab(path: Path, vocab: dict[str, int], embedding_dim: int = 100) -> dict[str, np.ndarray]:
    require_file(path)
    wanted = set(vocab.keys())
    embeddings = {}
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            values = line.rstrip().split(" ")
            word = values[0]
            if word not in wanted:
                continue
            vector = np.asarray(values[1:], dtype="float32")
            if vector.shape[0] == embedding_dim:
                embeddings[word] = vector
    return embeddings

def build_glove_matrix(vocab: dict[str, int], glove_embeddings: dict[str, np.ndarray], embedding_dim: int) -> np.ndarray:
    rng = np.random.default_rng(SEED)
    matrix = rng.normal(0, 0.05, size=(len(vocab), embedding_dim)).astype("float32")
    matrix[vocab[PAD_TOKEN]] = 0.0
    hits = 0
    for word, idx in vocab.items():
        vector = glove_embeddings.get(word)
        if vector is not None:
            matrix[idx] = vector
            hits += 1
    print(f"Embedding coverage: {hits} matched tokens ({hits / max(1, len(vocab)):.2%})")
    return matrix

glove_embeddings = load_glove_for_vocab(GLOVE_PATH, vocab, GLOVE_DIM)
glove_embedding_matrix = build_glove_matrix(vocab, glove_embeddings, GLOVE_DIM)

In [ ]:
glove_train_features = recurrent_features(X_train, glove_embedding_matrix, hidden_units=96, recurrent_scale=0.35, seed=SEED)
glove_test_features = recurrent_features(X_test, glove_embedding_matrix, hidden_units=96, recurrent_scale=0.35, seed=SEED)

glove_classifier = train_linear_head(glove_train_features, y_train)
glove_predictions = glove_classifier.predict(glove_test_features)
glove_metrics = evaluate_predictions(y_test, glove_predictions, "GloVe NumPy RNN")
glove_metrics

## Strong Classical Baseline: TF-IDF + Logistic Regression

This baseline is not one of the two required neural models, but it is useful for interpretation. TF-IDF often works very well for sentiment classification because it directly captures discriminative words and phrases.

Including this model helps answer an important experimental question: does the recurrent representation actually help on this noisy tweet dataset, or does a simpler sparse text model perform better?

In [ ]:
tfidf_model = make_pipeline(
    TfidfVectorizer(
        max_features=50000,
        ngram_range=(1, 2),
        min_df=2,
        sublinear_tf=True,
    ),
    LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        n_jobs=-1,
        random_state=SEED,
    ),
)

tfidf_model.fit(train_df["clean_text"], y_train)
tfidf_predictions = tfidf_model.predict(test_df["clean_text"])
tfidf_metrics = evaluate_predictions(y_test, tfidf_predictions, "TF-IDF Logistic Regression")
tfidf_metrics

## Model Comparison

In [ ]:
comparison_df = pd.DataFrame([custom_metrics, glove_metrics, tfidf_metrics]).sort_values("macro_f1", ascending=False)
display(comparison_df)

ax = comparison_df.set_index("model")[["accuracy", "macro_f1", "weighted_f1"]].plot(kind="bar", figsize=(9, 5))
ax.set_title("Model comparison")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

## Final Summary and Most Interesting Insights

1. **Macro F1 is the most important score.** It treats all five sentiment classes equally, which matters because the dataset is not perfectly balanced.

2. **Preprocessing has high impact.** Removing URLs and mentions reduces noise, while preserving hashtag words keeps useful pandemic and sentiment cues.

3. **The custom RNN encoder is lightweight and transparent.** It shows how sequential hidden states transform tweet text into vector features without relying on TensorFlow or PyTorch.

4. **GloVe transfer learning changes the input representation.** The transfer model starts from semantic word vectors learned on a large external corpus, then trains the classifier for COVID sentiment labels.

5. **The TF-IDF baseline is a reality check.** If it beats the lightweight RNN encoders, the insight is that sparse word and phrase features are very strong for this dataset, especially when the recurrent weights are not fully trained end-to-end.

6. **Extreme sentiment classes are likely hardest.** `Extremely Positive` and `Extremely Negative` are close to their non-extreme neighbors, so errors often happen between adjacent sentiment intensities.

7. **Future improvement:** on a machine with a stable deep-learning backend, a fully trainable BiLSTM/BiGRU with backpropagation-through-time or a tweet-specific transformer such as BERTweet would likely perform better.